In [6]:
# =============================================================================
# STEP 5 - WHAT THE NSL-KDD LADDER ACTUALLY MANIPULATES
#
# The ladder is the paper's primary evidence, so it deserves the hardest test.
# The committed subtype tables already show something the manuscript does not say:
#
#   the R2L source calibration pool is 86.6% warezclient, which has ZERO target
#   instances; the R2L target is 42.7% guess_passwd and 32.7% warezmaster, which
#   have 11 and 3 source instances. guess_passwd is counted as "seen" by S_sup and
#   its coverage is 0.029, as low as any genuinely unseen subtype.
#
# So rung 0.00 is not a no-subtype-shift condition, and S_sup's binary in-source
# test does not measure what matters. This notebook asks which of three competing
# quantities actually predicts per-subtype coverage:
#
#   (a) is_unseen        - the binary the ladder manipulates
#   (b) source support   - how many calibration points the subtype has
#   (c) score movement   - the mechanism the paper claims
#
# and then runs the three sensitivity checks the ladder requires: a lower-rung-only
# slope, a composition-adjusted regression, and leave-one-subtype-out.
#
# This can go against the paper. If (a) or (b) beats (c), the mechanism claim is
# weaker than stated and Sections 5.3 and 5.4 need restating rather than caveating.
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
from conformal import conformal_q
import numpy as np, pandas as pd
from scipy import stats
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY
FOCAL='R2L'
print('ready | alpha', ALPHA)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ready | alpha 0.05


In [7]:
# =============================================================================
# Cell 2 - assemble per-subtype data: source support, target mass, coverage and
# score movement, per rung and model.
# =============================================================================
def aps_mid(P):
    o=np.argsort(-P,axis=1); sp=np.take_along_axis(P,o,1); cum=np.cumsum(sp,1)
    ss=cum-0.5*sp; out=np.empty_like(P); np.put_along_axis(out,o,ss,1); return out

CL=config.CANONICAL_CLASSES; c2i={c:i for i,c in enumerate(CL)}; FIDX=c2i[FOCAL]
tr=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
te=pd.read_parquet(config.INTERIM_DIR/'nslkdd_test.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
tr=tr.assign(partition=part['partition'].values)
src=tr[tr.partition=='source_cal_pool'].reset_index(drop=True)
y_sp=src['label'].map(c2i).to_numpy(); sub_sp=src['subtype'].to_numpy()
y_te=te['label'].map(c2i).to_numpy(); sub_te=te['subtype'].to_numpy()

# source support per focal subtype, from the calibration pool actually used
supp=pd.Series(sub_sp[y_sp==FIDX]).value_counts().to_dict()
focal_subs_target=sorted(set(sub_te[y_te==FIDX]))
print(f'{FOCAL} source calibration pool: {int((y_sp==FIDX).sum())} points across '
      f'{len(supp)} subtypes')
print('  support:', {k:int(v) for k,v in sorted(supp.items(), key=lambda kv:-kv[1])})
print(f'  target subtypes: {len(focal_subs_target)}')
print('  target mass:', {s:int((sub_te[y_te==FIDX]==s).sum()) for s in focal_subs_target})

assign=pd.read_parquet(config.PROC_DIR/'nslkdd_ladder_assignments.parquet')
IDX={(r,j,role):g['test_idx'].to_numpy() for (r,j,role),g in assign.groupby(['rung','realization','role'])}
RUNGS=sorted(assign['rung'].unique()); REALS=sorted(assign['realization'].unique())
print(f'\nrungs {RUNGS} | realizations {len(REALS)}')

rows=[]
for f in sorted(glob.glob(str(config.PROC_DIR/'probs_*.npz'))):
    arch,seed=Path(f).stem.replace('probs_','').rsplit('_s',1)
    d=np.load(f); S_sp=aps_mid(d['S_pool'].astype(np.float64)); P_te=d['target'].astype(np.float64)
    src_scores=S_sp[y_sp==FIDX, FIDX]
    q,_=conformal_q(src_scores, ALPHA)
    if not np.isfinite(q): continue
    for rg in RUNGS:
        for j in REALS:
            ev=IDX.get((rg,j,'eval'))
            if ev is None or len(ev)==0: continue
            S_ev=aps_mid(P_te[ev]); ye=y_te[ev]; se=sub_te[ev]
            fm=ye==FIDX
            for s in np.unique(se[fm]):
                sm=fm&(se==s); n=int(sm.sum())
                if n<3: continue
                sc=S_ev[sm, FIDX]
                rows.append({'rung':float(rg),'realization':int(j),'arch':arch,'seed':seed,
                    'subtype':s,'n_target':n,'n_source':int(supp.get(s,0)),
                    'is_unseen':bool(supp.get(s,0)==0),
                    'coverage':float((sc<=q).mean()),
                    'score_KS':float(stats.ks_2samp(src_scores, sc).statistic),
                    'median_shift':float(np.median(sc)-np.median(src_scores))})
ST=pd.DataFrame(rows)
print(f'\nper-subtype cells: {len(ST):,} across {ST.subtype.nunique()} subtypes')


R2L source calibration pool: 149 points across 6 subtypes
  support: {'warezclient': 129, 'guess_passwd': 11, 'warezmaster': 3, 'ftp_write': 3, 'multihop': 2, 'imap': 1}
  target subtypes: 13
  target mass: {'ftp_write': 3, 'guess_passwd': 1231, 'httptunnel': 133, 'imap': 1, 'multihop': 18, 'named': 17, 'phf': 2, 'sendmail': 14, 'snmpgetattack': 178, 'snmpguess': 331, 'warezmaster': 944, 'xlock': 9, 'xsnoop': 4}

rungs [np.float64(0.0), np.float64(0.2), np.float64(0.4), np.float64(0.6), np.float64(0.8)] | realizations 20

per-subtype cells: 13,260 across 9 subtypes


In [8]:
# =============================================================================
# Cell 3 - THE COMPETING PREDICTORS. Which of the three orders per-subtype
# coverage? Aggregated to the subtype level so the model panel is not counted as
# independent replication.
# =============================================================================
sub=ST.groupby('subtype',as_index=False).agg(
    n_target=('n_target','mean'), n_source=('n_source','first'),
    is_unseen=('is_unseen','first'), coverage=('coverage','mean'),
    score_KS=('score_KS','mean'), median_shift=('median_shift','mean'))
sub=sub.sort_values('n_target',ascending=False)
print('R2L SUBTYPES, aggregated over rungs and models:')
print(sub.round(4).to_string(index=False))

print('\nWHICH PREDICTS PER-SUBTYPE COVERAGE?')
res={}
for name,x,direction in [('is_unseen (binary)', sub.is_unseen.astype(float), 'lower coverage expected'),
                         ('source support n_source', sub.n_source.astype(float), 'higher support -> higher coverage'),
                         ('log(1+source support)', np.log1p(sub.n_source.astype(float)), 'same, compressed'),
                         ('score movement (KS)', sub.score_KS, 'more movement -> lower coverage')]:
    if pd.Series(x).nunique()<2: print(f'  {name:26s} constant, skipped'); continue
    r,p=stats.spearmanr(x, sub.coverage)
    res[name]={'rho':round(float(r),3),'p':round(float(p),4)}
    print(f'  {name:26s} rho={r:+.3f}  p={p:.4f}   ({direction})')

print('\n  NOTE: n is small (one row per subtype), so these are indicative, not decisive.')
print('  The comparison that matters is the ORDERING of the three, not their significance.')

# the specific counterexample the composition table exposes
gp=sub[sub.subtype=='guess_passwd']; wm=sub[sub.subtype=='warezmaster']
if len(gp) and len(wm):
    print('\n  Counterexample to the support story:')
    print(f'    guess_passwd  support {int(gp.n_source.iloc[0]):2d}  coverage {float(gp.coverage.iloc[0]):.4f}  KS {float(gp.score_KS.iloc[0]):.3f}')
    print(f'    warezmaster   support {int(wm.n_source.iloc[0]):2d}  coverage {float(wm.coverage.iloc[0]):.4f}  KS {float(wm.score_KS.iloc[0]):.3f}')
    print('    warezmaster has LESS source support but HIGHER coverage, so raw support does')
    print('    not order the outcome; score movement is the candidate that can.')


R2L SUBTYPES, aggregated over rungs and models:
      subtype  n_target  n_source  is_unseen  coverage  score_KS  median_shift
    snmpguess  115.6964         0       True    0.0000    0.9860        0.5042
 guess_passwd  100.5700        11      False    0.0072    0.9722        0.5037
  warezmaster   75.3200         3      False    0.2431    0.8015        0.5009
snmpgetattack   58.6512         0       True    0.0021    0.9852        0.5042
   httptunnel   49.0800         0       True    0.0024    0.9777        0.5040
        named    6.4839         0       True    0.0047    0.9756        0.5037
     sendmail    4.7429         0       True    0.0248    0.9692        0.5039
     multihop    3.3333         2      False    0.0373    0.9618        0.5022
        xlock    3.0000         0       True    0.0000    0.9873        0.5039

WHICH PREDICTS PER-SUBTYPE COVERAGE?
  is_unseen (binary)         rho=-0.733  p=0.0245   (lower coverage expected)
  source support n_source    rho=+0.676  p=0.0

In [9]:
# =============================================================================
# Cell 4 - the three sensitivity checks the ladder requires.
# =============================================================================
import statsmodels.formula.api as smf
cov=pd.read_csv(RD/'coverage_primary_nslkdd.csv')
cov=cov[(cov['class']==FOCAL)&(np.isclose(cov.alpha,ALPHA))&(cov.protocol=='SHC')].copy()
cov['elogit']=np.log((cov.n_covered+0.5)/(cov.n_eval-cov.n_covered+0.5))
cov['rung_c']=cov['rung']-cov['rung'].mean()
cov['realization']=cov['realization'].astype(str)

print('(1) FULL-LADDER SLOPE (as published)')
m0=smf.mixedlm('elogit ~ rung_c', cov, groups=cov['realization']).fit(reml=True)
print(f'    slope {m0.fe_params["rung_c"]:+.4f}  se {m0.bse_fe["rung_c"]:.4f}  n={len(cov)}')

print('\n(2) LOWER-RUNGS-ONLY SLOPE (0.00 to 0.40, before the dominant subtype is forced)')
low=cov[cov['rung']<=0.40]
m1=smf.mixedlm('elogit ~ rung_c', low, groups=low['realization']).fit(reml=True)
print(f'    slope {m1.fe_params["rung_c"]:+.4f}  se {m1.bse_fe["rung_c"]:.4f}  n={len(low)}')
print(f'    ratio to full-ladder slope: {float(m1.fe_params["rung_c"]/m0.fe_params["rung_c"]):.3f}')
print('    a ratio near 1 means the effect is not created by the upper rungs')

print('\n(3) COMPOSITION-ADJUSTED: does the rung survive controlling for what is IN the eval set?')
def _composition(g):
    tot=g.n_target.sum()
    return pd.Series({
      'frac_unseen': float((g.n_target*g.is_unseen).sum()/tot),
      'wmean_support': float((g.n_target*g.n_source).sum()/tot),
      'frac_guess_passwd': float(g.loc[g.subtype=='guess_passwd','n_target'].sum()/tot),
      'frac_warezmaster': float(g.loc[g.subtype=='warezmaster','n_target'].sum()/tot)})
try:
    comp=ST.groupby(['rung','realization']).apply(_composition, include_groups=False).reset_index()
except TypeError:                      # pandas < 2.2 has no include_groups
    comp=ST.groupby(['rung','realization']).apply(_composition).reset_index()
# cov['realization'] was cast to str for the mixed model while comp carries it as int,
# so align the key dtypes before merging rather than after.
comp['realization']=comp['realization'].astype(str)
comp['rung']=comp['rung'].astype(float)
cov2=cov.copy(); cov2['rung']=cov2['rung'].astype(float)
cov2=cov2.merge(comp, on=['rung','realization'], how='left')
assert cov2['wmean_support'].notna().any(), 'merge produced no matches; check the key dtypes'
print(f'    merged composition onto {int(cov2.wmean_support.notna().sum())} of {len(cov2)} coverage rows')
for formula,label in [('elogit ~ rung_c','rung only'),
                      ('elogit ~ rung_c + wmean_support','+ weighted source support'),
                      ('elogit ~ rung_c + frac_guess_passwd + frac_warezmaster','+ dominant-subtype shares'),
                      ('elogit ~ wmean_support','support only, no rung')]:
    try:
        mm=smf.mixedlm(formula, cov2.dropna(subset=['wmean_support']), groups=cov2.dropna(subset=['wmean_support'])['realization']).fit(reml=True)
        terms={k:f'{v:+.4f}' for k,v in mm.fe_params.items() if k!='Intercept'}
        print(f'    {label:32s} {terms}')
    except Exception as e:
        print(f'    {label:32s} FAILED: {e}')

print('\n(4) LEAVE-ONE-SUBTYPE-OUT: recompute the focal coverage trend with each subtype removed')
loso=[]
for s in sorted(ST.subtype.unique()):
    g=ST[ST.subtype!=s]
    if g.rung.nunique()<3: continue
    byr=g.groupby('rung').apply(lambda d: float((d.coverage*d.n_target).sum()/d.n_target.sum())).astype(float)
    sl,_,r,p,_=stats.linregress(byr.index.astype(float), byr.values)
    loso.append({'dropped':s,'slope':round(float(sl),4),'r':round(float(r),3),'p':round(float(p),4)})
L=pd.DataFrame(loso).sort_values('slope')
print(L.to_string(index=False))
byr_all=ST.groupby('rung').apply(lambda d: float((d.coverage*d.n_target).sum()/d.n_target.sum())).astype(float)
sl0,_,r0,p0,_=stats.linregress(byr_all.index.astype(float), byr_all.values)
print(f'    with all subtypes: slope {sl0:+.4f} r {r0:+.3f} p {p0:.4f}')
print(f'    slope range across LOSO: {L.slope.min():+.4f} to {L.slope.max():+.4f}')
print('    if no single subtype flips the sign, the trend is not one subtype in disguise')


(1) FULL-LADDER SLOPE (as published)
    slope -2.0705  se 0.0297  n=3000

(2) LOWER-RUNGS-ONLY SLOPE (0.00 to 0.40, before the dominant subtype is forced)
    slope -1.4747  se 0.0612  n=1800
    ratio to full-ladder slope: 0.712
    a ratio near 1 means the effect is not created by the upper rungs

(3) COMPOSITION-ADJUSTED: does the rung survive controlling for what is IN the eval set?


/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


    merged composition onto 3000 of 3000 coverage rows
    rung only                        {'rung_c': '-2.0705'}
    + weighted source support        {'rung_c': '-3.1754', 'wmean_support': '-0.1464'}


/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


    + dominant-subtype shares        {'rung_c': '+1.2300', 'frac_guess_passwd': '+2.7842', 'frac_warezmaster': '+4.0691'}
    support only, no rung            {'wmean_support': '+0.2733'}

(4) LEAVE-ONE-SUBTYPE-OUT: recompute the focal coverage trend with each subtype removed
      dropped   slope      r      p
 guess_passwd -0.2609 -0.979 0.0035
     multihop -0.1065 -1.000 0.0000
     sendmail -0.1058 -1.000 0.0000
        xlock -0.1058 -1.000 0.0000
        named -0.1054 -1.000 0.0000
   httptunnel -0.1044 -0.995 0.0005
snmpgetattack -0.1029 -0.996 0.0003
    snmpguess -0.0734 -0.960 0.0097
  warezmaster -0.0084 -0.981 0.0030
    with all subtypes: slope -0.1058 r -1.000 p 0.0000
    slope range across LOSO: -0.2609 to -0.0084
    if no single subtype flips the sign, the trend is not one subtype in disguise


/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/tmp/ipykernel_1080/1791000830.py:58: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  byr=g.groupby('rung').apply(lambda d: float((d.coverage*d.n_target).sum()/d.n_target.sum())).astype(float)
/tmp/ipykernel_1080/1791000830.py:58: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicit

In [ ]:
# =============================================================================
# Cell 5 - save and commit.
# =============================================================================
ST.to_csv(RD/'nsl_subtype_cells.csv', index=False)
sub.to_csv(RD/'nsl_subtype_level.csv', index=False)
L.to_csv(RD/'nsl_leave_one_subtype_out.csv', index=False)
(RD/'nsl_ladder_sensitivity.json').write_text(json.dumps({
 'motivation':'the R2L source pool is 86.6% warezclient (zero target instances) while the '
              'target is 42.7% guess_passwd and 32.7% warezmaster (11 and 3 source instances), '
              'so rung 0.00 is not a no-subtype-shift condition and S_sup does not measure '
              'support adequacy',
 'predictor_comparison':res,
 'full_slope':float(m0.fe_params['rung_c']),'full_se':float(m0.bse_fe['rung_c']),
 'lower_rung_slope':float(m1.fe_params['rung_c']),'lower_rung_se':float(m1.bse_fe['rung_c']),
 'loso_slope_range':[float(L.slope.min()),float(L.slope.max())],
 'all_subtypes_slope':float(sl0)}, indent=2, default=str))
print('saved subtype cells, subtype-level table, LOSO and the sensitivity record')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','step 5: NSL ladder sensitivity; tests whether novelty, source support or score movement predicts per-subtype coverage')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


saved subtype cells, subtype-level table, LOSO and the sensitivity record
